In [3]:
import pandas as pd

FILE_PATH = r"C:\Users\ramin\Documents\Responsible Machine Learning\Capstone\2024_lar.txt"

chunks = []
for chunk in pd.read_csv(FILE_PATH, sep="|", chunksize=500_000, low_memory=False):
    chunk = chunk[chunk["action_taken"].isin([1, 2, 3])]
    chunks.append(chunk)
    print(f"Chunk loaded: {len(chunk):,} rows kept")

raw_data = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows: {raw_data.shape[0]:,}")
raw_data.head(5)

Chunk loaded: 291,927 rows kept
Chunk loaded: 266,002 rows kept
Chunk loaded: 289,477 rows kept
Chunk loaded: 360,009 rows kept
Chunk loaded: 324,596 rows kept
Chunk loaded: 365,079 rows kept
Chunk loaded: 371,797 rows kept
Chunk loaded: 377,231 rows kept
Chunk loaded: 436,906 rows kept
Chunk loaded: 365,741 rows kept
Chunk loaded: 173,496 rows kept
Chunk loaded: 364,466 rows kept
Chunk loaded: 387,767 rows kept
Chunk loaded: 373,944 rows kept
Chunk loaded: 364,975 rows kept
Chunk loaded: 362,575 rows kept
Chunk loaded: 389,483 rows kept
Chunk loaded: 371,027 rows kept
Chunk loaded: 264,950 rows kept
Chunk loaded: 386,274 rows kept
Chunk loaded: 414,145 rows kept
Chunk loaded: 372,687 rows kept
Chunk loaded: 472,770 rows kept
Chunk loaded: 344,904 rows kept
Chunk loaded: 169,549 rows kept

Total rows: 8,661,777


,activity_year,lei,derived_msa_md,state_code,county_code,census_tract,conforming_loan_limit,derived_loan_product_type,derived_dwelling_category,derived_ethnicity,...,denial_reason_2,denial_reason_3,denial_reason_4,tract_population,tract_minority_population_percent,ffiec_msa_md_median_family_income,tract_to_msa_income_percentage,tract_owner_occupied_units,tract_one_to_four_family_homes,tract_median_age_of_housing_units
0,2024,5493007I0X1GRWIU8B34,36260,UT,49011.0,49011126304.0,C,Conventional:Subordinate Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,6955,11.78,110200,143.0,2000,2163,30
1,2024,5493007I0X1GRWIU8B34,41100,UT,49053.0,49053271702.0,C,Conventional:Subordinate Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,15965,14.72,101200,124.0,2504,3464,12
2,2024,5493007I0X1GRWIU8B34,36260,UT,49057.0,49057210303.0,C,Conventional:Subordinate Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,8164,11.73,110200,125.0,1997,2096,29
3,2024,5493007I0X1GRWIU8B34,36260,UT,49057.0,49057201500.0,C,Conventional:Subordinate Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,4261,18.52,110200,96.0,1056,1196,61
4,2024,5493007I0X1GRWIU8B34,29820,NV,32003.0,32003002916.0,C,Conventional:Subordinate Lien,Single Family (1-4 Units):Site-Built,Not Hispanic or Latino,...,NaN,NaN,NaN,2839,42.30,87800,105.0,546,968,28


In [4]:
print(list(raw_data.columns))

['activity_year', 'lei', 'derived_msa_md', 'state_code', 'county_code', 'census_tract', 'conforming_loan_limit', 'derived_loan_product_type', 'derived_dwelling_category', 'derived_ethnicity', 'derived_race', 'derived_sex', 'action_taken', 'purchaser_type', 'preapproval', 'loan_type', 'loan_purpose', 'lien_status', 'reverse_mortgage', 'open_end_line_of_credit', 'business_or_commercial_purpose', 'loan_amount', 'combined_loan_to_value_ratio', 'interest_rate', 'rate_spread', 'hoepa_status', 'total_loan_costs', 'total_points_and_fees', 'origination_charges', 'discount_points', 'lender_credits', 'loan_term', 'prepayment_penalty_term', 'intro_rate_period', 'negative_amortization', 'interest_only_payment', 'balloon_payment', 'other_nonamortizing_features', 'property_value', 'construction_method', 'occupancy_type', 'manufactured_home_secured_property_type', 'manufactured_home_land_property_interest', 'total_units', 'multifamily_affordable_units', 'income', 'debt_to_income_ratio', 'applicant_cre

In [5]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

# Select only the columns we need
COLS = [
    "action_taken",
    "loan_amount",
    "income",
    "property_value",
    "interest_rate",
    "loan_term",
    "combined_loan_to_value_ratio",
    "debt_to_income_ratio",
    "loan_type",
    "loan_purpose",
    "lien_status",
    "occupancy_type",
    "applicant_age",
    "derived_race",
    "derived_sex",
    "derived_ethnicity",
    "state_code",
]

df = raw_data[COLS].copy()

# Label creation per assignment spec
df = df[df["action_taken"].isin([1, 2, 3])].copy()
df["label"] = df["action_taken"].map({1: 1, 2: 1, 3: 0})
df = df.drop(columns=["action_taken"])

print(f"Rows after filtering : {df.shape[0]:,}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nApproval rate: {df['label'].mean():.3f}")

Rows after filtering : 8,661,777

Label distribution:
label
1    6558317
0    2103460
Name: count, dtype: int64

Approval rate: 0.757


In [6]:
# Sample 200k rows for modeling (stratified on label)
df = df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(frac=200_000 / len(df), random_state=42)
).reset_index(drop=True)

print(f"Sampled rows  : {df.shape[0]:,}")
print(f"Approval rate : {df['label'].mean():.3f}")

Sampled rows  : 200,000
Approval rate : 0.757


In [7]:
# EDA — race distribution
race_dist = df["derived_race"].value_counts(normalize=True) * 100
for race, pct in race_dist.items():
    print(f"{race}: {pct:.2f}%")

White: 64.07%
Race Not Available: 17.67%
Black or African American: 8.84%
Asian: 5.92%
Joint: 2.25%
American Indian or Alaska Native: 0.73%
Native Hawaiian or Other Pacific Islander: 0.25%
2 or more minority races: 0.25%
Free Form Text Only: 0.02%


In [8]:
# EDA — sex distribution
sex_dist = df["derived_sex"].value_counts(normalize=True) * 100
for sex, pct in sex_dist.items():
    print(f"{sex}: {pct:.2f}%")

Joint: 34.51%
Male: 34.16%
Female: 22.45%
Sex Not Available: 8.88%


In [9]:
# Approval rate by race
print("── Approval rate by race ──")
print(df.groupby("derived_race")["label"].mean().round(3).sort_values())

── Approval rate by race ──
derived_race
Free Form Text Only                          0.532
2 or more minority races                     0.554
Native Hawaiian or Other Pacific Islander    0.609
American Indian or Alaska Native             0.634
Black or African American                    0.635
Race Not Available                           0.724
White                                        0.781
Asian                                        0.794
Joint                                        0.806
Name: label, dtype: float64


In [10]:
# Approval rate by sex
print("── Approval rate by sex ──")
print(df.groupby("derived_sex")["label"].mean().round(3).sort_values())

── Approval rate by sex ──
derived_sex
Female               0.708
Sex Not Available    0.734
Male                 0.738
Joint                0.814
Name: label, dtype: float64


## Converting data

In [15]:
# Convert numeric columns — replace non-numeric values with NaN
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

X = df[numeric_features + categorical_features]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {X_train.shape[0]:,}")
print(f"Test size  : {X_test.shape[0]:,}")

Train size : 160,000
Test size  : 40,000


In [16]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [17]:
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

gbt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(n_estimators=200, random_state=42)),
])

lr_pipeline.fit(X_train, y_train)
print("Logistic Regression trained.")

gbt_pipeline.fit(X_train, y_train)
print("Gradient Boosted Tree trained.")

Logistic Regression trained.
Gradient Boosted Tree trained.


In [18]:
for name, pipeline in [("Logistic Regression", lr_pipeline), ("Gradient Boosted Tree", gbt_pipeline)]:
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    print(f"\n── {name} ──")
    print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
    print(f"AUC       : {roc_auc_score(y_test, y_prob):.3f}")
    print(f"Log-loss  : {log_loss(y_test, y_prob):.3f}")
    print(f"Precision : {tp / (tp + fp):.3f}")
    print(f"Recall    : {tp / (tp + fn):.3f}")
    print(f"FPR       : {fp / (fp + tn):.3f}")
    print(f"FNR       : {fn / (fn + tp):.3f}")


── Logistic Regression ──
Accuracy  : 0.826
AUC       : 0.819
Log-loss  : 0.408
Precision : 0.832
Recall    : 0.966
FPR       : 0.609
FNR       : 0.034

── Gradient Boosted Tree ──
Accuracy  : 0.975
AUC       : 0.997
Log-loss  : 0.061
Precision : 0.984
Recall    : 0.983
FPR       : 0.049
FNR       : 0.017


In [19]:
def group_metrics(pipeline, X, y, group_col):
    results = []
    X_ = X.copy().reset_index(drop=True)
    y_ = y.reset_index(drop=True)
    X_["actual"]    = y_
    X_["pred"]      = pipeline.predict(X)
    X_["pred_prob"] = pipeline.predict_proba(X)[:, 1]

    for group, gdf in X_.groupby(group_col):
        tn, fp, fn, tp = confusion_matrix(
            gdf["actual"], gdf["pred"], labels=[0, 1]
        ).ravel()

        def safe_div(num, den):
            return num / den if den != 0 else np.nan

        results.append({
            "group"        : group,
            "n"            : len(gdf),
            "approval_rate": round(gdf["pred"].mean(), 3),
            "accuracy"     : round(accuracy_score(gdf["actual"], gdf["pred"]), 3),
            "FPR"          : round(safe_div(fp, fp + tn), 3),
            "FNR"          : round(safe_div(fn, fn + tp), 3),
            "AUC"          : round(
                roc_auc_score(gdf["actual"], gdf["pred_prob"])
                if gdf["actual"].nunique() > 1 else np.nan, 3
            ),
        })

    return pd.DataFrame(results).sort_values("n", ascending=False).reset_index(drop=True)

print("── Logistic Regression — by Race ──")
print(group_metrics(lr_pipeline, X_test, y_test, "derived_race"))

print("\n── Gradient Boosted Tree — by Race ──")
print(group_metrics(gbt_pipeline, X_test, y_test, "derived_race"))

── Logistic Regression — by Race ──
                                       group      n  approval_rate  accuracy  \
0                                      White  25659          0.901     0.839   
1                         Race Not Available   7091          0.861     0.794   
2                  Black or African American   3460          0.758     0.758   
3                                      Asian   2388          0.897     0.878   
4                                      Joint    902          0.915     0.847   
5           American Indian or Alaska Native    290          0.728     0.793   
6                   2 or more minority races    111          0.631     0.793   
7  Native Hawaiian or Other Pacific Islander     89          0.697     0.775   
8                        Free Form Text Only     10          0.500     0.600   

     FPR    FNR    AUC  
0  0.640  0.026  0.811  
1  0.620  0.045  0.817  
2  0.500  0.095  0.793  
3  0.548  0.016  0.847  
4  0.678  0.030  0.812  
5  0.416  0.0

In [20]:
def compute_air(pipeline, X, group_col, reference_group):
    X_ = X.copy().reset_index(drop=True)
    X_["pred"] = pipeline.predict(X)
    rates = X_.groupby(group_col)["pred"].mean()
    ref_rate = rates[reference_group]
    air = (rates / ref_rate).round(3)
    return air.sort_values().reset_index()

print("── AIR by Race (reference = White) — Logistic Regression ──")
print(compute_air(lr_pipeline, X_test, "derived_race", "White"))

print("\n── AIR by Race (reference = White) — Gradient Boosted Tree ──")
print(compute_air(gbt_pipeline, X_test, "derived_race", "White"))

── AIR by Race (reference = White) — Logistic Regression ──
                                derived_race   pred
0                        Free Form Text Only  0.555
1                   2 or more minority races  0.700
2  Native Hawaiian or Other Pacific Islander  0.773
3           American Indian or Alaska Native  0.808
4                  Black or African American  0.842
5                         Race Not Available  0.956
6                                      Asian  0.996
7                                      White  1.000
8                                      Joint  1.016

── AIR by Race (reference = White) — Gradient Boosted Tree ──
                                derived_race   pred
0                        Free Form Text Only  0.639
1                   2 or more minority races  0.656
2  Native Hawaiian or Other Pacific Islander  0.718
3           American Indian or Alaska Native  0.780
4                  Black or African American  0.803
5                         Race Not Available 